# Diplomado de Grado en Análisis de Datos Aplicado a la Toma de Decisiones
## UNICOLOMBO — Fundación Universitaria Colombo Internacional · Cartagena
**Módulo 2:** Preparación y Análisis Exploratorio de Datos  
**Sesión 4:** EDA e Insights de Negocio  
**Docente:** Ing. Heyder Medrano Olier | **Período:** 2026 — Vacaciones

---
**Tipo:** Notebook de Actividad  
**Entrega:** 2 de julio de 2026 · 23:59 (hora Colombia)  
**Valor:** 5.0 puntos  

**Estudiante:** Jefferson Arrieta Duran

> Realice el EDA completo del dataset DataRetail LATAM. Incluya visualizaciones, pruebas estadísticas y mínimo 5 insights de negocio propios.

## 1. Librerías

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import scipy.stats as stats, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('OK')

## 2. Cargar Dataset

In [ ]:
import numpy as np, pandas as pd, random, scipy.stats as stats
from datetime import datetime, timedelta
np.random.seed(42); random.seed(42)
N=2000
ciudades=["Bogotá","Medellín","Cali","Barranquilla","Cartagena",
          "Lima","Ciudad de México","Buenos Aires","Santiago","Montevideo"]
paises=["Colombia","Perú","México","Argentina","Chile","Uruguay"]
cats=["Computación","Periféricos","Audio","Telefonía","Accesorios","Gaming","Componentes"]
canales=["Tienda Física","E-commerce","Distribuidor","Corporativo","Marketplace"]
prods=["Laptop Pro","Tablet Ultra","Monitor 4K","Teclado Inalámbrico",
       "Auriculares BT","Parlante Portátil","Smartphone Galaxy",
       "Cámara Web HD","Smartwatch Pro","Mouse Gaming","SSD 1TB","GPU RTX 4060"]
precios={"Laptop Pro":1200,"Tablet Ultra":450,"Monitor 4K":380,"Teclado Inalámbrico":75,
         "Auriculares BT":120,"Parlante Portátil":90,"Smartphone Galaxy":680,
         "Cámara Web HD":95,"Smartwatch Pro":250,"Mouse Gaming":65,"SSD 1TB":130,"GPU RTX 4060":580}
rows=[]
for i in range(N):
    prod=random.choice(prods); cat=cats[prods.index(prod)%len(cats)]
    pais=random.choice(paises); ciudad=random.choice(ciudades); canal=random.choice(canales)
    qty=random.randint(1,20); precio=round(precios[prod]*np.random.uniform(0.8,1.3),2)
    desc=round(random.choice([0,0,0,0.05,0.10,0.15,0.20,0.25,0.30]),2)
    total=round(qty*precio*(1-desc),2); margen=round(total*np.random.uniform(0.05,0.35),2)
    fecha=datetime(2024,1,1)+timedelta(days=random.randint(0,729))
    rows.append({"id_venta":f"V{i+1:05d}","fecha_venta":fecha,"id_cliente":f"C{random.randint(1,500):04d}",
                 "nombre_cliente":f"Cliente {random.randint(1,500)}","ciudad":ciudad,"pais":pais,
                 "id_producto":f"P{prods.index(prod)+1:02d}","nombre_producto":prod,"categoria":cat,
                 "canal_venta":canal,"cantidad":qty,"precio_unitario":precio,"descuento":desc,
                 "total_venta":total,"margen_utilidad":margen})
df = pd.DataFrame(rows)
df["margen_pct"] = (df["margen_utilidad"]/df["total_venta"]).round(4)
df["anio"] = df["fecha_venta"].dt.year
df["mes"]  = df["fecha_venta"].dt.month
df["trimestre"] = df["fecha_venta"].dt.quarter
df["dia_semana"] = df["fecha_venta"].dt.dayofweek
print(f"Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas (limpio+transformado)")
df.head(3)


## 3. Análisis Univariado — Numéricas

In [ ]:
cols_n=['cantidad','precio_unitario','descuento','total_venta','margen_utilidad']
desc = df[cols_n].describe(percentiles=[.05,.25,.50,.75,.95]).T
desc['skewness'] = df[cols_n].skew()
print(desc[['mean','std','min','50%','max','skewness']].to_string())

In [ ]:
cols_n = ['cantidad','precio_unitario','descuento','total_venta','margen_utilidad']

desc = df[cols_n].describe(percentiles=[.05,.25,.50,.75,.95]).T

desc['skewness'] = df[cols_n].skew()
desc['kurtosis'] = df[cols_n].kurt()

print(
    desc[
        ['mean','std','min','50%','max','skewness','kurtosis']
    ].to_string()
)

In [ ]:
 for col in cols_n:
    plt.figure(figsize=(7,3))

    sns.boxplot(
        x=df[col],
        color="red",
        width=0.4
    )

    plt.title(f'Boxplot - {col}', fontsize=12)
    plt.xlabel(col)
    plt.grid(axis='x', linestyle='--', alpha=0.6)

    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribuciones Numéricas — EDA S04 (Mejorado)', fontweight='bold')

for ax, col in zip(axes.flatten(), cols_n):

    s = df[col].dropna()

    # Histograma
    ax.hist(s, bins=40, density=True, alpha=0.6, edgecolor='white')

    # KDE
    s.plot.kde(ax=ax, color='navy', lw=2)

    # MEDIA
    mean = s.mean()
    ax.axvline(mean, color='green', linestyle='--', label='Media')

    # MEDIANA
    median = s.median()
    ax.axvline(median, color='purple', linestyle='--', label='Mediana')

    # OUTLIERS (IQR)
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    ax.axvspan(lower, upper, color='yellow', alpha=0.1, label='Rango sin outliers')

    # Título con skew
    ax.set_title(f'{col} (skew={s.skew():.2f})', fontweight='bold')

    ax.legend(fontsize=7)

# Apagar subplot sobrante
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

## 4. Histogramas con KDE

In [ ]:
fig, axes = plt.subplots(2,3,figsize=(15,8))
fig.suptitle('Distribuciones Numéricas — EDA S04', fontweight='bold')
for i,(ax,col) in enumerate(zip(axes.flatten(),cols_n)):
    s=df[col].dropna()
    ax.hist(s,bins=40,density=True,alpha=0.7,ec='white')
    s.plot.kde(ax=ax,color='navy',lw=2)
    ax.axvline(s.mean(),color='red',linestyle='--',label=f'Media')
    ax.set_title(f'{col} (skew={s.skew():.2f})',fontweight='bold')
    ax.legend(fontsize=7)
axes[1,2].axis('off')
plt.tight_layout(); plt.show()

## 5. Análisis Univariado — Categóricas

In [ ]:
fig,axes = plt.subplots(2,2,figsize=(14,9))
for ax,col in zip(axes.flatten(),['categoria','canal_venta','pais','ciudad']):
    df[col].value_counts().plot(kind='bar',ax=ax,ec='white')
    ax.set_title(col,fontweight='bold'); ax.tick_params(axis='x',rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 9))
fig.suptitle('Distribución de Variables Categóricas', fontsize=15, fontweight='bold')

categoricas = ['categoria', 'canal_venta', 'pais', 'ciudad']
axes = axes.ravel()

for i in range(len(categoricas)):

    frecuencia = df[categoricas[i]].value_counts().nlargest(10)

    axes[i].barh(
        frecuencia.index[::-1],
        frecuencia.values[::-1],
        edgecolor='black'
    )

    axes[i].set_title(f'{categoricas[i]}', fontweight='bold')
    axes[i].set_xlabel('Cantidad de registros')
    axes[i].set_ylabel('')

plt.tight_layout()
plt.show()

## 6. Ventas por Canal y Categoría (Pivote)

In [ ]:
pivot=df.pivot_table(values='total_venta',index='categoria',columns='canal_venta',aggfunc='sum',fill_value=0)
pivot.plot(kind='bar',figsize=(12,5),colormap='Set2',ec='white')
plt.title('Total Ventas por Categoría y Canal',fontweight='bold')
plt.xticks(rotation=45); plt.legend(title='Canal',bbox_to_anchor=(1.02,1))
plt.tight_layout(); plt.show()
print(pivot.to_string())

Este análisis facilita la comprensión del desempeño comercial al mostrar qué categorías de productos concentran un mayor volumen de ventas y cuáles son los canales de comercialización que generan mejores resultados dentro de cada una de ellas. Además, permite identificar tendencias y diferencias en el comportamiento de las ventas según la combinación de categoría y canal. En el contexto de la inteligencia de negocios, esta información resulta valiosa para detectar oportunidades de mejora, fortalecer las estrategias de comercialización y orientar la toma de decisiones con base en datos.

##Interpretación de resultados (Pivot de ventas)
La tabla dinámica permite analizar cómo se distribuye el valor total de las ventas entre las diferentes categorías de productos y los canales de venta. Gracias a esta organización de la información, es posible reconocer patrones comerciales, comparar el desempeño de cada categoría en los distintos canales e identificar cuáles generan una mayor contribución a las ventas de la empresa.

## 7. Matriz de Correlaciones

In [ ]:
corr=df[cols_n].corr()
fig,ax=plt.subplots(figsize=(8,6))
mask=np.triu(np.ones_like(corr,dtype=bool))
sns.heatmap(corr,annot=True,fmt='.3f',cmap='RdBu_r',center=0,mask=mask,ax=ax,linewidths=0.5)
ax.set_title('Correlaciones Pearson — EDA S04',fontweight='bold')
plt.tight_layout(); plt.show()

Justificación del análisis de correlación

El propósito de este análisis es evaluar el grado de relación existente entre las variables numéricas del conjunto de datos. Para ello, se emplea el coeficiente de correlación de Pearson, el cual permite medir la intensidad y la dirección de la relación lineal entre las variables. Los resultados se presentan mediante un mapa de calor (heatmap), facilitando la identificación de aquellas variables que muestran una mayor asociación y que pueden resultar relevantes para el análisis exploratorio y el desarrollo de modelos predictivos.

📝 Interpretación de la correlación más fuerte

La correlación de mayor magnitud encontrada en la matriz evidencia una relación positiva significativa entre las variables analizadas. Esto significa que, a medida que una variable incrementa su valor, la otra también presenta una tendencia de crecimiento. En el contexto de este conjunto de datos, este comportamiento es lógico, ya que variables como total_venta y margen_utilidad están estrechamente vinculadas al resultado económico de cada transacción.

Desde la perspectiva del negocio, esta relación sugiere que un incremento en el valor de las ventas suele estar acompañado de un aumento en la utilidad obtenida, lo que refleja un comportamiento consistente en la operación comercial. Además, esta fuerte asociación permite identificar variables clave para el análisis del desempeño financiero y constituye una base importante para desarrollar modelos predictivos y diseñar estrategias orientadas a mejorar la rentabilidad y el crecimiento de la empresa.

## 8. Análisis Temporal

In [ ]:
vm=df.groupby(['anio','mes'])['total_venta'].sum().reset_index()
vm['fecha'] = pd.to_datetime(vm['anio'].astype(str) + '-' + vm['mes'].astype(str) + '-01')
vm['ma3']=vm['total_venta'].rolling(3,min_periods=1).mean()
fig,ax=plt.subplots(figsize=(14,4))
ax.fill_between(vm['fecha'],vm['total_venta'],alpha=0.3)
ax.plot(vm['fecha'],vm['total_venta'],lw=1.5,label='Ventas')
ax.plot(vm['fecha'],vm['ma3'],color='red',lw=2.5,linestyle='--',label='MA-3')
ax.set_title('Tendencia Mensual',fontweight='bold'); ax.legend()
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

📝 ¿Se observa estacionalidad en las ventas?

El comportamiento de las ventas a lo largo de los meses evidencia la presencia de cierta estacionalidad, ya que existen periodos en los que la actividad comercial aumenta y otros en los que disminuye. Estas variaciones responden a los hábitos de consumo de los clientes y a eventos comerciales que se repiten cada año.

Los mayores niveles de ventas suelen presentarse hacia el cierre del año, principalmente durante noviembre y diciembre, meses en los que se desarrollan campañas como Black Friday, Cyber Monday y las compras de la temporada navideña. Asimismo, en algunos casos también pueden registrarse incrementos al comienzo del año debido a promociones y descuentos posteriores a las festividades.

En contraste, durante algunos meses del periodo intermedio es posible observar una reducción en las ventas, ya que, después de temporadas de alto consumo, la demanda suele estabilizarse y los clientes disminuyen su ritmo de compra.

En términos generales, este comportamiento refleja que las ventas no mantienen un ritmo uniforme durante todo el año, sino que están influenciadas por ciclos de consumo y fechas comerciales relevantes en Latinoamérica. Identificar estos patrones resulta útil para planificar campañas de marketing, gestionar inventarios y definir estrategias comerciales en los periodos de mayor y menor demanda.

## 9. Análisis de Pareto

In [ ]:
pv=df.groupby('nombre_producto')['total_venta'].sum().sort_values(ascending=False)
pp=(pv/pv.sum()*100).cumsum()
fig,ax1=plt.subplots(figsize=(12,5))
ax1.bar(pv.index,pv.values,color='#4e79a7',ec='white')
ax2=ax1.twinx()
ax2.plot(pv.index,pp.values,color='#e15759',lw=2,marker='o',ms=5)
ax2.axhline(80,color='green',linestyle='--')
ax1.set_title('Análisis Pareto — Ventas por Producto',fontweight='bold')
ax1.tick_params(axis='x',rotation=45); plt.tight_layout(); plt.show()
n80=(pp<=80).sum()
print(f'{n80} productos generan el 80% de las ventas')

## 10. Segmentación RFM

In [ ]:
fr=df['fecha_venta'].max()
rfm=df.groupby('id_cliente').agg(
    recencia=('fecha_venta',lambda x:(fr-x.max()).days),
    frecuencia=('id_venta','count'),monetario=('total_venta','sum')
).reset_index()
rfm['R_score']=pd.qcut(rfm['recencia'],q=4,labels=[4,3,2,1]).astype(int)
rfm['F_score']=pd.qcut(rfm['frecuencia'],q=4,labels=[1,2,3,4]).astype(int)
rfm['M_score']=pd.qcut(rfm['monetario'],q=4,labels=[1,2,3,4]).astype(int)
rfm['RFM']=rfm['R_score']+rfm['F_score']+rfm['M_score']
rfm['segmento']=rfm['RFM'].apply(lambda x:'Champions' if x>=10 else ('Loyal' if x>=7 else ('Potential' if x>=5 else 'At Risk')))
print(rfm.groupby('segmento').agg(n=('id_cliente','count'),monetario_med=('monetario','median')).to_string())

## 11. Hipótesis de Negocio (mínimo 2 pruebas)

H1: El promedio de las ventas de la categoría Computación es significativamente mayor que el promedio de las ventas de la categoría Audio.

Interpretación de los resultados:

Los resultados muestran que la categoría Computación registró un promedio de ventas de $5.712,10, mientras que la categoría Audio alcanzó un promedio de 2.284,38. Al realizar la prueba de hipótesis se obtuvo un valor p = 0.0000, el cual es menor que el nivel de significancia establecido (α = 0.05). Con base en este resultado, se rechaza la hipótesis nula, concluyendo que la diferencia entre los promedios de ventas de ambas categorías es estadísticamente significativa. Esto indica que, dentro del conjunto de datos analizado, la categoría Computación genera, en promedio, ventas superiores a las de la categoría Audio, por lo que esta diferencia difícilmente puede atribuirse al azar.



In [ ]:
# Complete con su hipótesis y la prueba estadística
# Ejemplo: ¿Computación tiene mayor ticket que Audio?
comp  = df[df['categoria']=='Computación']['total_venta'].dropna()
audio = df[df['categoria']=='Audio']['total_venta'].dropna()
t,p = stats.ttest_ind(comp,audio)
print(f'Computación: ${comp.mean():.2f} | Audio: ${audio.mean():.2f}')
print(f'p-value: {p:.4f} → {"SIGNIFICATIVO" if p<0.05 else "no significativo"} (alpha=0.05)')

📝 Interpretación H1

Con base en el resultado obtenido, se rechaza la hipótesis nula, ya que el valor p es inferior al nivel de significancia establecido. Esto confirma que existe una diferencia estadísticamente significativa entre el promedio de ventas de las categorías Computación y Audio, siendo la primera la que presenta un mejor desempeño.

Implicaciones para el negocio: Este hallazgo indica que la categoría Computación tiene una mayor contribución a los ingresos de la empresa. Por ello, puede ser conveniente fortalecer las estrategias comerciales relacionadas con esta línea de productos, optimizar la gestión del inventario y orientar campañas de promoción hacia esta categoría, con el fin de potenciar las ventas y mejorar la rentabilidad del negocio.

### H2:  Existe una asociación significativa entre la categoría del producto y el canal de venta.

In [ ]:
# Su segunda prueba estadística aquí
# Hint: chi2_contingency para variables categóricas
tabla = pd.crosstab(df['categoria'], df['canal_venta'])
chi2, p_chi, dof, _ = stats.chi2_contingency(tabla)
print(f'Chi²={chi2:.4f} | p={p_chi:.6f} | dof={dof}')

📝 Interpretación H2

Para comprobar esta hipótesis se utilizó la prueba de Chi-cuadrado de independencia, obteniéndose un estadístico Chi² = 27.2884, un valor p = 0.291190 y 24 grados de libertad.

Dado que el valor p es superior al nivel de significancia de 0.05, no se rechaza la hipótesis nula. En consecuencia, los resultados no proporcionan evidencia suficiente para afirmar que existe una relación significativa entre la categoría del producto y el canal de venta.

Desde la perspectiva del negocio, esto sugiere que la elección del canal de venta no depende de manera significativa de la categoría del producto, por lo que ambas variables parecen comportarse de forma independiente dentro de los datos analizados.



### Mis 5+ Insights de Negocio

### Insight 1: Los canales digitales representan una oportunidad de crecimiento

Crecimiento

Los canales E-commerce y Marketplace muestran un desempeño positivo, pero aún tienen una participación menor que la Tienda Física en la mayoría de las categorías. Esto indica que existe espacio para aumentar las ventas digitales mediante campañas promocionales, mejoras en la experiencia de compra en línea y estrategias de fidelización.

### Insight 2: El canal físico sigue siendo un pilar del negocio

Alto impacto

La Tienda Física continúa concentrando los mayores niveles de ventas en categorías como Computación, Componentes y Accesorios. Este resultado confirma que el canal presencial mantiene una alta relevancia comercial y que fortalecer la disponibilidad de productos y la experiencia del cliente puede generar un impacto directo en los ingresos.

### Insight 3: Computación lidera la generación de ingresos

Prioridad estratégica

La categoría Computación presenta el mayor volumen de ventas dentro del portafolio y supera ampliamente a otras categorías en distintos canales. Debido a su contribución al negocio, se recomienda priorizar inversiones en inventario, marketing y estrategias comerciales orientadas a mantener y ampliar su participación en los ingresos.

### Insight 4: Gaming muestra el menor desempeño comercial

Revisión recomendada

La categoría Gaming registra los niveles de ventas más bajos en todos los canales analizados. Este comportamiento puede reflejar una demanda limitada o una menor presencia de productos, por lo que sería conveniente evaluar si se justifica impulsar esta línea mediante promociones específicas o concentrar los recursos en categorías con mayor rentabilidad.

### Insight 5: Las categorías mantienen un comportamiento similar entre canales

Chi-cuadrado

La prueba de Chi-cuadrado obtuvo un p-value = 0.291, valor superior al nivel de significancia de 0.05. Por tanto, no se encontró evidencia estadística suficiente para afirmar que exista una asociación significativa entre la categoría del producto y el canal de venta. Esto sugiere que las categorías presentan un comportamiento relativamente consistente entre los distintos canales, lo que facilita el diseño de estrategias comerciales generales para toda la operación.





## Referencias
1. Tukey, J.W. (1977). *Exploratory Data Analysis*. Addison-Wesley.
2. McKinney, W. (2022). *Python for Data Analysis* (3.ª ed.). O'Reilly.
3. Waskom, M. (2021). seaborn. *JOSS*, 6(60), 3021.
4. Hughes, A.M. (2005). *Strategic Database Marketing*. McGraw-Hill.
5. Virtanen, P. et al. (2020). SciPy 1.0. *Nature Methods*, 17, 261-272.

---
*Notebook Actividad S04 — UNICOLOMBO · 2026*